# Validación del modelo barotrópico con difusión

Este notebook valida el flujo modular para

\[
p=\omega\rho,\qquad Q=\alpha\rho^\beta,\qquad
3H^2=\rho+\alpha\rho^\beta.
\]

Se comprueban de Sitter, la rama F2, la
frontera \(D(\rho)=0\), el turnaround y el catálogo representativo B1–F4.
Todas las implementaciones se importan desde `src`.


In [1]:
from pathlib import Path
import sys

CURRENT_PATH = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (CURRENT_PATH, *CURRENT_PATH.parents)
        if (candidate / "src").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró la raíz del proyecto.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Entorno del proyecto localizado correctamente.")


Entorno del proyecto localizado correctamente.


In [2]:
import importlib
import numpy as np
import pandas as pd

import src.models as models_module
import src.barotropic_analysis as analysis_module

importlib.reload(models_module)
importlib.reload(analysis_module)

from src.models import BarotropicPowerDiffusionParams
from src.barotropic_analysis import (
    analyze_barotropic_case,
    compute_critical_densities,
    infer_boundary_evidence,
    infer_limit_evidence,
    infer_signed_collapse_evidence,
    initial_density_direction,
    scan_boundary_tolerances,
    scan_density_targets,
    scan_signed_collapse_thresholds,
    solve_signed_background_compatible,
    summarize_signed_background,
)
print("Importaciones completadas.")


Importaciones completadas.


## 1. Configuración numérica

In [3]:
FAST_MODE = True
NUMERICS = {
    "t_end": 20.0 if FAST_MODE else 50.0,
    "n_points": 1800 if FAST_MODE else 5000,
    "max_step": 0.01 if FAST_MODE else 0.001,
    "density_decades": 6 if FAST_MODE else 10,
    "boundary_tolerances": [1e-2,1e-3,1e-4,1e-5,1e-6] if FAST_MODE else [1e-2,1e-3,1e-4,1e-5,1e-6,1e-7,1e-8],
    "collapse_rho_max_values": [1e2,1e3,1e4,1e5,1e6] if FAST_MODE else [1e2,1e3,1e4,1e5,1e6,1e7,1e8],
}
NUMERICS


{'t_end': 20.0,
 'n_points': 1800,
 'max_step': 0.01,
 'density_decades': 6,
 'boundary_tolerances': [0.01, 0.001, 0.0001, 1e-05, 1e-06],
 'collapse_rho_max_values': [100.0, 1000.0, 10000.0, 100000.0, 1000000.0]}

## 2. de Sitter exacto

In [4]:
params_de_sitter = BarotropicPowerDiffusionParams(
    omega=-1.0, alpha=0.3, beta=1.0, rho0=1.0, a0=1.0, t0=0.0
)
analysis_de_sitter = analyze_barotropic_case(
    params_de_sitter, case_id="B2_de_sitter", **NUMERICS
)
de_sitter_result = analysis_de_sitter.classification.as_dict()
de_sitter_result


{'standard_type': 'Non singular',
 'behavior_label': 'de_sitter',
 'status': 'robust',
 'is_singularity': False,
 'finite_time': False,
 'confidence': 0.95,
 'observed_behavior': {'time_limit': 'unknown',
  'rho': 'unknown',
  'a': 'unknown',
  'H': 'unknown',
  'Hdot': 'unknown',
  'rho_dot': 'unknown',
  'Hddot': 'unknown',
  'termination_event': None,
  'solver_success': True,
  'reached_t_end': True,
  'H_zero_crossings': 0,
  'expansion_to_contraction': False,
  'initial_H_sign': 1,
  'final_H_sign': 1,
  'initial_values': {'t': 0.0,
   'rho': 1.0,
   'a': 1.0,
   'H': 0.6582805886043833,
   'Hdot': -0.0},
  'final_values': {'t': 20.0,
   'rho': 1.0,
   'a': 522098.61784039007,
   'H': 0.6582805886043833,
   'Hdot': -0.0},
  'tail_relative_spread': {'rho': 0.0,
   'a': 0.7301731203663804,
   'H': 0.0,
   'Hdot': 0.0},
  'p': 'unknown'},
 'evidence': {'stable_under_threshold_changes': True, 'confidence': 0.95},
 'notes': 'rho y H permanecen constantes y dot(H)=0.'}

In [5]:
assert de_sitter_result["behavior_label"] == "de_sitter"
assert de_sitter_result["is_singularity"] is False
print("de Sitter exacto: OK")


de Sitter exacto: OK


## 3. Rama F2 dominada por difusión

In [6]:
params_F2 = BarotropicPowerDiffusionParams(
    omega=-1.2, alpha=4.0, beta=-1.0, rho0=1.0, a0=1.0, t0=0.0
)
print(compute_critical_densities(params_F2).as_dict())
print("Dirección inicial:", initial_density_direction(params_F2))


{'rho_F': None, 'rho_D': 2.0, 'F_degenerate': False, 'D_degenerate': False, 'notes': ''}
Dirección inicial: -1


In [7]:
targets_F2 = [10.0**(-k) for k in range(1, NUMERICS["density_decades"]+1)]
scan_F2 = scan_density_targets(params_F2, targets_F2, require_future=True, strict=False)
evidence_F2 = infer_limit_evidence(scan_F2)
display(scan_F2[["success","rho","t","log_a","H","Hdot","rho_dot","Hddot"]])
evidence_F2


,success,rho,t,log_a,H,Hdot,rho_dot,Hddot
0,True,0.100000,1.150202e+02,3.261624e+02,3.656045,1.000000e-02,-5.497812e-04,-5.497812e-05
1,True,0.010000,3.841370e+03,3.332232e+04,11.547150,1.000000e-03,-1.732116e-06,-1.732116e-07
2,True,0.001000,1.217082e+05,3.333318e+06,36.514842,1.000000e-04,-5.477228e-09,-5.477228e-10
3,True,0.000100,3.848994e+06,3.333333e+08,115.470054,1.000000e-05,-1.732051e-11,-1.732051e-12
4,True,0.000010,1.217161e+08,3.333333e+10,365.148372,1.000000e-06,-5.477226e-14,-5.477226e-15
5,True,0.000001,3.849002e+09,3.333333e+12,1154.700538,1.000000e-07,-1.732051e-16,-1.732051e-17


{'time_limit': 'infinite',
 'rho_limit': 'zero',
 'a_limit': 'diverges',
 'H_limit': 'unknown',
 'Hdot_limit': 'unknown',
 'rho_dot_limit': 'zero',
 'Hddot_limit': 'zero',
 'stable_under_threshold_changes': True,
 'confidence': 0.85,
 'target_direction': 'decay_to_zero',
 'estimated_limit_time': inf,
 'diagnostics': {'time': {'time_limit': 'infinite',
   'estimated_limit_time': inf,
   'time_increment_ratio': 31.622777388103902,
   'time_convergence_stable': True},
  'a': {'limit': 'diverges',
   'step_ratio': 100.00000115129258,
   'total_log_change': 3333333332980.813,
   'last_log_step': 3299999999996.1636},
  'H': {'limit': 'unknown',
   'ratio': 315.83322092747215,
   'log_slope': 0.49999998366023113,
   'tail_relative_spread': 0.9683772194454734,
   'increasing_fraction': 1.0,
   'decreasing_fraction': 0.0},
  'Hdot': {'limit': 'unknown',
   'ratio': 9.999999999999999e-06,
   'log_slope': -0.9999999999999999,
   'tail_relative_spread': 0.999,
   'increasing_fraction': 0.0,
   'de

In [8]:
numerics_F2 = {
    **NUMERICS,
    "density_decades": 10,
}

analysis_F2 = analyze_barotropic_case(
    params=params_F2,
    case_id="F2",
    **numerics_F2,
)

result_F2 = (
    analysis_F2
    .classification
    .as_dict()
)

print(
    "Etiqueta:",
    result_F2["behavior_label"],
)

print(
    "Límite temporal:",
    result_F2["finite_time"],
)

print(
    "Evidencia:"
)

display(
    pd.Series(
        analysis_F2.evidence
    )
)

assert (
    result_F2["behavior_label"]
    == "diffusion_driven_infinite_time_rip"
), result_F2

assert (
    result_F2["finite_time"]
    is False
), result_F2

print("Prueba F2: OK")

Etiqueta: diffusion_driven_infinite_time_rip
Límite temporal: False
Evidencia:


time_limit                                                                 infinite
rho_limit                                                                      zero
a_limit                                                                    diverges
H_limit                                                                    diverges
Hdot_limit                                                                     zero
rho_dot_limit                                                                  zero
Hddot_limit                                                                    zero
stable_under_threshold_changes                                                 True
confidence                                                                     0.95
target_direction                                                      decay_to_zero
estimated_limit_time                                                            inf
diagnostics                       {'time': {'time_limit': 'infinite', 'estim

Prueba F2: OK


## 4. Frontera D(rho)=0 y tipo IV

In [9]:
params_D = BarotropicPowerDiffusionParams(
    omega=0.0, alpha=0.25, beta=-1.0, rho0=1.0, a0=1.0, t0=0.0
)
print(compute_critical_densities(params_D).as_dict())
scan_D = scan_boundary_tolerances(
    params_D,
    t_end=NUMERICS["t_end"],
    boundary_tolerances=NUMERICS["boundary_tolerances"],
    n_points=NUMERICS["n_points"],
    rho_min=1e-10,
    rho_max=1e10,
    a_max=1e50,
    max_step=min(NUMERICS["max_step"], 0.001),
)
evidence_D = infer_boundary_evidence(scan_D)
display(scan_D[["boundary_tol","termination_event","t_final","rho_final","a_final","H_final","Hdot_final","D_final","rho_dot_final","Hddot_final"]])
evidence_D


{'rho_F': None, 'rho_D': 0.5, 'F_degenerate': False, 'D_degenerate': False, 'notes': ''}


,boundary_tol,termination_event,t_final,rho_final,a_final,H_final,Hdot_final,D_final,rho_dot_final,Hddot_final
0,0.010000,rho_dot_denominator_boundary,0.174354,0.502519,1.111867,0.577354,-0.251259,1.000000e-02,-87.039377,43.519689
1,0.001000,rho_dot_denominator_boundary,0.174368,0.500250,1.111876,0.577350,-0.250125,1.000000e-03,-866.458796,433.229398
2,0.000100,rho_dot_denominator_boundary,0.174368,0.500025,1.111876,0.577350,-0.250013,1.000000e-04,-8660.687026,4330.343513
3,0.000010,rho_dot_denominator_boundary,0.174368,0.500003,1.111876,0.577350,-0.250001,1.000001e-05,-86602.913401,43301.456700
4,0.000001,rho_dot_denominator_boundary,0.174368,0.500000,1.111876,0.577350,-0.250000,9.992819e-07,-866648.179908,433324.089954


{'time_limit': 'finite',
 'rho_limit': 'finite',
 'a_limit': 'finite',
 'H_limit': 'finite',
 'Hdot_limit': 'finite',
 'rho_dot_limit': 'diverges',
 'Hddot_limit': 'diverges',
 'stable_under_tolerance_changes': True,
 'confidence': 0.95,
 'boundary_type': 'D',
 'estimated_limit_time': 0.1743682258939886,
 'estimated_boundary_density': 0.5000092505721597,
 'estimated_log_a': 0.10604905990577558,
 'diagnostics': {'time': {'converges': True,
   'limit': 0.1743682258939886,
   'relative_spread': 1.4433352291121082e-09},
  'rho': {'converges': True,
   'limit': 0.5000092505721597,
   'relative_spread': 2.475205467455943e-05},
  'log_a': {'converges': True,
   'limit': 0.10604905990577558,
   'relative_spread': 8.333100043333985e-10},
  'H_finite': {'converges': True,
   'limit': 0.5773502693111339,
   'relative_spread': 3.6084391030755114e-10},
  'H_zero': {'tends_to_zero': False,
   'log_slope': -5.510614754491702e-07,
   'intercept': -0.23855786928027933,
   'decay_ratio': 0.9999936869683

In [10]:
analysis_D = analyze_barotropic_case(
    params_D,
    case_id="D_boundary_type_IV",
    t_end=NUMERICS["t_end"],
    n_points=NUMERICS["n_points"],
    max_step=min(NUMERICS["max_step"],0.001),
    density_decades=NUMERICS["density_decades"],
    boundary_tolerances=NUMERICS["boundary_tolerances"],
    collapse_rho_max_values=NUMERICS["collapse_rho_max_values"],
)
result_D = analysis_D.classification.as_dict()
assert result_D["standard_type"] == "Type IV"
assert result_D["behavior_label"] == "higher_derivative_boundary"
result_D


{'standard_type': 'Type IV',
 'behavior_label': 'higher_derivative_boundary',
 'status': 'robust',
 'is_singularity': True,
 'finite_time': True,
 'confidence': 0.95,
 'observed_behavior': {'time_limit': 'finite',
  'rho': 'finite',
  'a': 'finite',
  'H': 'finite',
  'Hdot': 'finite',
  'rho_dot': 'diverges',
  'Hddot': 'diverges',
  'termination_event': 'rho_dot_denominator_boundary',
  'solver_success': True,
  'reached_t_end': False,
  'H_zero_crossings': 0,
  'expansion_to_contraction': False,
  'initial_H_sign': 1,
  'final_H_sign': 1,
  'initial_values': {'t': 0.0,
   'rho': 1.0,
   'a': 1.0,
   'H': 0.6454972243679028,
   'Hdot': -0.5},
  'final_values': {'t': 0.17436822637986363,
   'rho': 0.5000002498206613,
   'a': 1.1118764240329648,
   'H': 0.5773502691896618,
   'Hdot': -0.25000012491033063},
  'tail_relative_spread': {'rho': 0.16837727400614005,
   'a': 0.01000782791926227,
   'H': 0.00841538062778267,
   'Hdot': 0.16837727400614005},
  'p': 'zero'},
 'evidence': {'time_

## 5. Turnaround y Big Crunch

In [11]:
params_turnaround = BarotropicPowerDiffusionParams(
    omega=0.0, alpha=-0.5, beta=0.5, rho0=1.0, a0=1.0, t0=0.0
)
print(compute_critical_densities(params_turnaround).as_dict())
signed = solve_signed_background_compatible(
    params_turnaround,
    t_end=NUMERICS["t_end"],
    n_points=max(NUMERICS["n_points"],2500),
    rho_min=1e-12,
    rho_max=1e6,
    a_min=1e-12,
    a_max=1e100,
    H_max=1e100,
    boundary_tol=1e-8,
    max_step=min(NUMERICS["max_step"],0.001),
)
summary_signed = summarize_signed_background(signed)
assert summary_signed["turnaround_detected"] is True
assert summary_signed["contracting_at_end"] is True
summary_signed


{'rho_F': 0.25, 'rho_D': 0.0625, 'F_degenerate': False, 'D_degenerate': False, 'notes': ''}


{'turnaround_detected': True,
 'turnaround_time': 1.905158688787075,
 'turnaround_density': 0.2500000900024124,
 'turnaround_scale_factor': 1.3437058999387077,
 'maximum_a': 1.3437059251251724,
 'time_at_maximum_a': 1.9043462104365219,
 'initial_H': 0.408248290463863,
 'final_H': -577.2059135675152,
 'final_rho': 999999.9999997154,
 'final_a': 0.011811635358704855,
 'final_Hdot': -499999.9999998577,
 'contracting_at_end': True,
 'termination_event': 'rho_max',
 'solver_success': True,
 'reached_t_end': False,
 'maximum_constraint_residual': 2.269175923875375e-08}

In [12]:
collapse_scan = scan_signed_collapse_thresholds(
    params_turnaround,
    t_end=NUMERICS["t_end"],
    rho_max_values=NUMERICS["collapse_rho_max_values"],
    n_points=max(NUMERICS["n_points"],2500),
    rho_min=1e-12,
    a_min=1e-12,
    a_max=1e100,
    H_max=1e100,
    boundary_tol=1e-8,
    max_step=min(NUMERICS["max_step"],0.001),
)
collapse_evidence = infer_signed_collapse_evidence(collapse_scan)
display(collapse_scan[["rho_max","turnaround_detected","termination_event","t_final","rho_final","a_final","H_final"]])
collapse_evidence


,rho_max,turnaround_detected,termination_event,t_final,rho_final,a_final,H_final
0,100.0,True,rho_max,4.868878,100.0,0.250310,-5.627314
1,1000.0,True,rho_max,4.947845,1000.0,0.117515,-18.112506
2,10000.0,True,rho_max,4.972813,10000.0,0.054743,-57.590508
3,100000.0,True,rho_max,4.980709,100000.0,0.025438,-182.429791
4,1000000.0,True,rho_max,4.983205,1000000.0,0.011812,-577.205914


{'time_limit': 'finite',
 'rho_limit': 'diverges',
 'a_limit': 'zero',
 'H_limit': 'diverges',
 'Hdot_limit': 'diverges',
 'rho_dot_limit': 'diverges',
 'Hddot_limit': 'diverges',
 'stable_under_threshold_changes': True,
 'confidence': 0.95,
 'turnaround_detected': True,
 'contracting_branch': True,
 'estimated_limit_time': 4.98436010195099,
 'diagnostics': {'time': {'time_limit': 'finite',
   'estimated_limit_time': 4.98436010195099,
   'time_increment_ratio': 0.31622351203468646,
   'time_convergence_stable': True},
  'rho': {'diverges': True,
   'log_slope': 0.9999999999999809,
   'intercept': 6.712772350380626e-14,
   'growth_factor': 9999.999999997166,
   'increasing_fraction': 1.0},
  'a_zero': {'tends_to_zero': True,
   'log_slope': -0.33169415758233795,
   'intercept': 0.0637152558262326,
   'decay_ratio': 0.04718810375129574,
   'decreasing_fraction': 1.0},
  'H': {'diverges': True,
   'log_slope': 0.502517641380292,
   'intercept': -0.25185913322486586,
   'growth_factor': 10

In [13]:
analysis_turnaround = analyze_barotropic_case(
    params_turnaround,
    case_id="D3",
    t_end=NUMERICS["t_end"],
    n_points=max(NUMERICS["n_points"],2500),
    max_step=min(NUMERICS["max_step"],0.001),
    density_decades=NUMERICS["density_decades"],
    boundary_tolerances=NUMERICS["boundary_tolerances"],
    collapse_rho_max_values=NUMERICS["collapse_rho_max_values"],
)
result_turnaround = analysis_turnaround.classification.as_dict()
assert result_turnaround["behavior_label"] in {"turnaround_to_big_crunch","turnaround_to_contraction"}
result_turnaround


{'standard_type': None,
 'behavior_label': 'turnaround_to_big_crunch',
 'status': 'robust',
 'is_singularity': True,
 'finite_time': True,
 'confidence': 0.95,
 'observed_behavior': {'time_limit': 'finite',
  'rho': 'diverges',
  'a': 'zero',
  'H': 'diverges',
  'Hdot': 'diverges',
  'rho_dot': 'diverges',
  'Hddot': 'diverges',
  'termination_event': 'rho_max',
  'solver_success': True,
  'reached_t_end': False,
  'H_zero_crossings': 1,
  'expansion_to_contraction': True,
  'initial_H_sign': 1,
  'final_H_sign': -1,
  'initial_values': {'t': 0.0,
   'rho': 1.0,
   'a': 1.0,
   'H': 0.408248290463863,
   'Hdot': -0.5},
  'final_values': {'t': 4.984358969808887,
   'rho': 1000000001522.3475,
   'a': 0.00011813602153727818,
   'H': -577350.1252987094,
   'Hdot': -500000000761.17377},
  'tail_relative_spread': {'rho': 0.9999999999945696,
   'a': 0.9998111981889127,
   'H': 0.9999979347433778,
   'Hdot': 0.9999999999945696},
  'maximum_constraint_residual': 1.0761121159072195e-07,
  'p': 

## 6. Catálogo B1–F4

In [14]:
CASES = {
"B1": (0.0,0.3,1.0), "B2": (-1.0,0.3,1.0), "B3": (-1.2,0.3,1.0),
"C1": (-1.2,0.3,1.5), "C2": (-1.2,0.3,2.0), "C3": (-1.2,0.3,3.0), "C4": (-1.2,-0.3,1.5),
"D1": (0.0,0.3,0.5), "D2": (-1.2,0.3,0.5), "D3": (0.0,-0.5,0.5),
"E1": (0.0,0.3,0.0), "E2": (0.0,-0.5,0.0),
"F1": (-1.2,0.25,-1.0), "F2": (-1.2,4.0,-1.0), "F3": (0.0,0.25,-1.0), "F4": (0.0,-0.5,-1.0),
}
REPRESENTATIVE_CASES = {
    key: BarotropicPowerDiffusionParams(omega=w, alpha=a, beta=b, rho0=1.0, a0=1.0, t0=0.0)
    for key,(w,a,b) in CASES.items()
}
len(REPRESENTATIVE_CASES)


16

In [15]:
catalog_analyses, catalog_errors = {}, {}
for case_id, params in REPRESENTATIVE_CASES.items():
    try:
        catalog_analyses[case_id] = analyze_barotropic_case(
            params,
            case_id=case_id,
            **NUMERICS,
        )
        print(case_id, "OK")
    except Exception as error:
        catalog_errors[case_id] = repr(error)
        print(case_id, "ERROR:", error)
print("Exitosos:", len(catalog_analyses), "Errores:", len(catalog_errors))


B1 OK


B2 OK


B3 OK


C1 OK


C2 OK


C3 OK


... (5 outputs left out)

E2 OK


F1 OK


F2 OK
F3 OK


F4 OK
Exitosos: 16 Errores: 0


In [16]:
catalog_summary = pd.DataFrame([
    item.summary_dict() for item in catalog_analyses.values()
]).sort_values("case_id").reset_index(drop=True)
display(catalog_summary)
catalog_errors


,case_id,omega,alpha,beta,rho0,rho_F,rho_D,initial_density_direction,analysis_route,termination_event,...,is_singularity,finite_time,confidence,time_limit,rho_limit,a_limit,H_limit,Hdot_limit,rho_dot_limit,Hddot_limit
0,B1,0.0,0.30,1.0,1.0,NaN,NaN,-1,density_limit,None,...,False,False,0.55,infinite,zero,unknown,unknown,unknown,zero,zero
1,B2,-1.0,0.30,1.0,1.0,NaN,NaN,0,stationary_density,None,...,False,False,0.95,None,None,None,None,None,None,None
2,B3,-1.2,0.30,1.0,1.0,NaN,NaN,1,density_limit,rho_max,...,True,True,0.90,finite,diverges,diverges,unknown,diverges,diverges,diverges
3,C1,-1.2,0.30,1.5,1.0,NaN,NaN,1,density_limit,a_max,...,True,True,0.95,finite,diverges,diverges,diverges,diverges,diverges,diverges
4,C2,-1.2,0.30,2.0,1.0,NaN,NaN,1,density_limit,a_max,...,False,False,0.95,infinite,diverges,diverges,diverges,diverges,diverges,diverges
5,C3,-1.2,0.30,3.0,1.0,NaN,NaN,1,density_limit,None,...,False,False,0.85,infinite,diverges,diverges,diverges,diverges,unknown,unknown
6,C4,-1.2,-0.30,1.5,1.0,11.111111,4.938272,1,boundary_D,rho_dot_denominator_boundary,...,True,True,0.95,finite,finite,finite,finite,finite,diverges,diverges
7,D1,0.0,0.30,0.5,1.0,NaN,NaN,-1,density_limit,None,...,False,False,0.85,infinite,zero,diverges,unknown,unknown,zero,zero
8,D2,-1.2,0.30,0.5,1.0,NaN,NaN,1,density_limit,rho_max,...,True,True,0.90,finite,diverges,diverges,unknown,diverges,diverges,diverges
9,D3,0.0,-0.50,0.5,1.0,0.250000,0.062500,-1,boundary_F,effective_density_boundary,...,True,True,0.95,finite,diverges,zero,diverges,diverges,diverges,diverges


{}

## 7. Controles estructurales y exportación

In [17]:
checks = {
    "at_least_12_cases": len(catalog_analyses) >= 12,
    "all_classified": all(x.classification is not None for x in catalog_analyses.values()),
    "all_backgrounds": all(x.background is not None for x in catalog_analyses.values()),
}
assert all(checks.values()), checks
checks


{'at_least_12_cases': True, 'all_classified': True, 'all_backgrounds': True}

In [18]:
assert not catalog_errors, catalog_errors
assert len(catalog_analyses) == len(REPRESENTATIVE_CASES)
print(
    f"Validación completada: {len(catalog_analyses)} casos representativos, "
    "sin errores de catálogo."
)


Validación completada: 16 casos representativos, sin errores de catálogo.


## Nota

Los estados `provisional`, `new_behavior_candidate` y `numerically_inconclusive` no deben reemplazarse manualmente. Indican que hace falta ampliar umbrales, tolerancias o análisis asintótico.
